# 06 — Criar o Registry e configurar a governança

Vamos criar e configurar os serviços durante as aulas, observar as
respostas da AWS e testar cada componente antes de avançar.

Referência: [workshops AgentCore](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops).

**Pré-requisitos:** uma conta AWS autorizada, região escolhida e o kernel
**Santander Workshop (.venv)** preparado por `scripts/setup.sh`.
Use seu ambiente de notebooks na AWS. Os serviços, ferramentas e agentes
desta jornada executam na AWS.

### Abrir uma sessão AWS e conferir a identidade

O SDK usa a sessão autorizada do seu ambiente AWS. STS informa quem está executando e em qual conta. A região é uma escolha da aula; ajuste-a antes de criar recursos. Nenhuma chave deve ser colada no notebook.

In [ ]:
import json
import os
import uuid
from pathlib import Path

import boto3
from IPython.display import display

REGION = boto3.Session().region_name or "us-west-2"
aws = boto3.Session(region_name=REGION)
sts = aws.client("sts")
identity = sts.get_caller_identity()
ACCOUNT_ID = identity["Account"]
expected_account = os.environ.get("WORKSHOP_EXPECTED_ACCOUNT")
if expected_account and ACCOUNT_ID != expected_account:
    raise RuntimeError(
        "WORKSHOP_ACCOUNT_MISMATCH: a sessão não corresponde à conta esperada."
    )
PARTITION = identity["Arn"].split(":")[1]
print({
    "account": ACCOUNT_ID,
    "region": REGION,
    "principal": identity["Arn"],
    "request_id": identity["ResponseMetadata"]["RequestId"],
})

### Registrar identificadores desta execução

As próximas aulas precisam lembrar os IDs retornados pelas chamadas de criação. Este registro contém apenas identificadores, está vinculado à conta/região e permite reiniciar o kernel. Ele é criado agora: você não precisa receber um arquivo privado do instrutor. O helper abaixo não chama AWS nem cria serviços.

In [ ]:
from santander_demo.notebook_state import NotebookState

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").is_file()
)
state = NotebookState(
    ROOT / ".workshop", account_id=ACCOUNT_ID, region=REGION,
)
NAME_PREFIX = state.prefix
TAGS = state.tags
print({
    "execution": state.data["run_id"],
    "resource_name_prefix": NAME_PREFIX,
    "recorded_resources": sorted(state.data["resources"]),
    "execution_status": state.data.get("status", "ACTIVE"),
})

### Um catálogo não concede acesso ao serviço catalogado

`Publicação → revisão → aprovação → descoberta → autenticação no serviço`

Vamos criar um Registry nesta conta, com descoberta autenticada por
IAM e aprovação manual. As próximas aulas acrescentam as personas
e os registros. Criar o catálogo não cria um Runtime nem concede
permissão para invocá-lo.

**Pré-requisito:** aula 00 e acesso às APIs Agent Registry na região.
Uma conta basta. Esta aula não exige Cognito, configuração privada
do instrutor ou um Registry anterior. A configuração de descoberta
por IAM é uma escolha explícita; Cognito/Entra ID são alternativas
de authorizer JWT, com suas próprias audiências e claims.

### Conferir os clientes de controle e descoberta

O cliente agent-registry-control administra o catálogo. O cliente agent-registry consulta registros aprovados. Conferimos as operações disponíveis no SDK antes de criar o recurso.

In [ ]:
import time
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError

registry_control = aws.client("agent-registry-control")
registry_discovery = aws.client("agent-registry")
for client, operations in (
    (registry_control, {"CreateRegistry", "GetRegistry", "ListTagsForResource", "ListRegistryRecords"}),
    (registry_discovery, {"ListDiscoverableRegistryRecords", "BatchGetDiscoverableRegistryRecord"}),
):
    missing = operations - set(client.meta.service_model.operation_names)
    assert not missing, "REGISTRY_SDK_OPERATIONS_MISSING: " + str(sorted(missing))
print({"region": REGION, "discovery_authentication": "AWS_IAM", "approval": "MANUAL"})

### Criar o Registry com aprovação manual

A lista vazia de autoApprovalRules exige aprovação dos registros. CreateRegistry retorna o ARN: extraímos o ID dessa resposta, guardamos ambos e aguardamos READY. A retomada exige o mesmo nome, tags, authorizer e configuração de aprovação.

In [ ]:
REGISTRY_NAME = NAME_PREFIX + "-registry"
registry_entry = state.get("registry.instance")
if registry_entry is None:
    created = registry_control.create_registry(
        name=REGISTRY_NAME,
        description="Workshop Santander: catalogo com revisao manual",
        discoveryConfiguration={"authorizerType": "AWS_IAM"},
        approvalConfiguration={"autoApprovalRules": []},
        clientToken=str(uuid.uuid5(uuid.NAMESPACE_URL, NAME_PREFIX + ":registry")),
        tags=TAGS,
    )
    registry_entry = state.record("registry.instance", {
        "arn": created["registryArn"],
        "id": created["registryArn"].rsplit("/", 1)[1],
        "name": REGISTRY_NAME,
    }, lesson="06")
deadline = time.monotonic() + 300
while True:
    catalog = registry_control.get_registry(registryId=registry_entry["id"])
    if catalog["status"] == "READY":
        break
    if catalog["status"] in {"CREATE_FAILED", "UPDATE_FAILED", "DELETE_FAILED", "DELETING"}:
        raise RuntimeError("REGISTRY_FAILED: " + catalog.get("statusReason", ""))
    if time.monotonic() >= deadline:
        raise TimeoutError("REGISTRY_NOT_READY")
    time.sleep(5)
assert catalog["registryArn"] == registry_entry["arn"]
assert catalog["registryId"] == registry_entry["id"]
assert catalog["name"] == registry_entry["name"] == REGISTRY_NAME
assert catalog["discoveryConfiguration"] == {"authorizerType": "AWS_IAM"}
assert not catalog.get("approvalConfiguration", {}).get("autoApprovalRules", [])
assert catalog.get("autoDetection", {}).get("status", "INACTIVE") == "INACTIVE"
tags = registry_control.list_tags_for_resource(resourceArn=registry_entry["arn"])["tags"]
assert all(tags.get(key) == value for key, value in TAGS.items()), "REGISTRY_OWNERSHIP_MISMATCH"
print({
    "registry_id": registry_entry["id"], "status": catalog["status"],
    "request_id": catalog["ResponseMetadata"]["RequestId"],
})

### Consultar o catálogo e comprovar a autenticação

A chamada assinada usa a identidade da sessão AWS. Fazemos também uma chamada sem assinatura: somente uma negativa de autenticação com Request ID AWS conta como resultado esperado. Um catálogo vazio recém-criado é normal; as aulas seguintes publicarão registros.

In [ ]:
records = []
args = {"registryId": registry_entry["id"], "maxResults": 100}
while True:
    page = registry_discovery.list_discoverable_registry_records(**args)
    records.extend(page["registryRecords"])
    if not page.get("nextToken"):
        break
    args["nextToken"] = page["nextToken"]
assert all(record["registryArn"] == registry_entry["arn"] for record in records)
print({
    "discovery": "AUTHENTICATED", "records": len(records),
    "request_id": page["ResponseMetadata"]["RequestId"],
})
unsigned = aws.client("agent-registry", config=Config(signature_version=UNSIGNED))
try:
    unsigned.list_discoverable_registry_records(registryId=registry_entry["id"])
except ClientError as error:
    metadata = error.response["ResponseMetadata"]
    assert metadata["HTTPStatusCode"] in {401, 403}
    assert metadata["RequestId"]
    assert error.response["Error"]["Code"] in {
        "AccessDeniedException", "UnauthorizedException",
        "UnrecognizedClientException", "MissingAuthenticationTokenException",
    }, "EXPECTED_REGISTRY_AUTHENTICATION_DENIAL"
    print({
        "anonymous_discovery": "DENY", "http_status": metadata["HTTPStatusCode"],
        "request_id": metadata["RequestId"],
    })
else:
    raise AssertionError("ANONYMOUS_REGISTRY_DISCOVERY_ALLOWED")

### O que demonstramos

O Registry foi criado durante a aula, está READY e exige IAM na
descoberta. A aprovação automática permanece desativada.
A descoberta autenticada foi exercitada e a chamada anônima foi negada.
Isso ainda não comprova publicação, aprovação ou invocação de um agente.

A aula 99 chama `DeleteRegistry` somente depois de verificar que
não existem registros ou associações desconhecidas. Ao acrescentar
registros nas aulas seguintes, a limpeza também precisa contemplá-los.